<a href="https://colab.research.google.com/github/carolotero/aprendiendo/blob/main/Agente_LangChain_Memoria_y_Tools_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AGENTE ReACT EN LANGCHAIN CON MEMORIA, TOOLS Y MCP

ReAct (Reasoning + Acting) es un patrón de agente que combina razonamiento y acción en un bucle iterativo. Ante cada pregunta, el agente piensa qué necesita para responder, decide si usar una tool o responder directamente, observa el resultado y repite el proceso hasta tener suficiente información. Es el patrón más usado en producción por su simplicidad y efectividad.

- **Memoria**: recuerda el contexto de la conversación entre turnos
- **Tools**: puede buscar en internet y hacer cálculos matemáticos
- **Razonamiento ReAct**: decide qué tool usar según la pregunta del usuario

## Librerías necesarias

Antes de empezar, instalamos las librerías necesarias. Cada una tiene un rol específico en la construcción del agente:

- **langchain**: Framework principal que orquesta agentes, tools y cadenas
- **langchain-google-genai**: Conecta LangChain con los modelos Gemini de Google
- **langgraph**: Motor que gestiona el flujo del agente y la memoria entre turnos
- **tavily-python**: Cliente oficial para usar la API de búsqueda web de Tavily
- **langchain-tavily**: Integración de Tavily como tool nativa de LangChain



In [ ]:
!pip install -q langchain langchain-google-genai langgraph tavily-python langchain-tavily

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 1.7 MB/s eta 0:00:00


## Importamos las librerías

Importamos todas las librerías necesarias antes de empezar. Cada una aporta una pieza específica al agente: el modelo de lenguaje, la gestión de mensajes, las tools y el motor que orquesta todo el flujo.



In [ ]:
# Librerías estándar de Python
import os
import warnings
from getpass import getpass

# Silencia logs de INFO y DEBUG mostrando solo errores críticos
warnings.filterwarnings("ignore")

# Librerías de LangChain y LangGraph
from langchain_google_genai import ChatGoogleGenerativeAI  # Conecta LangChain con los modelos Gemini de Google
from langchain_core.messages import HumanMessage  # Estructura los mensajes del usuario
from langchain_tavily import TavilySearch  # Tool de búsqueda web en tiempo real
from langgraph.prebuilt import create_react_agent  # Crea un agente ReAct con razonamiento y uso de tools
from langgraph.checkpoint.memory import MemorySaver  # Gestiona la memoria persistente entre turnos
from langchain_core.tools import Tool  # Wrapper para definir y configurar tools para el agente


## Autenticación

Para usar los modelos de Google y la API de Tavily necesitamos autenticarnos con nuestras claves, podemos encontrarlas en: https://aistudio.google.com/ y https://app.tavily.com/.

Usamos `getpass` para introducirlas de forma segura, ocultando el input al escribir y evitando que queden expuestas en el código.

In [ ]:
# AUTENTICACIÓN - Configuramos las API keys como variables de entorno de forma segura
os.environ["GOOGLE_API_KEY"] = getpass("Introduce tu Google API key: ")
os.environ["TAVILY_API_KEY"] = getpass("Introduce tu Tavily API key: ")



Introduce tu Google API key: ··········
Introduce tu Tavily API key: ··········
Introduce tu GitHub token: ··········


## 1. Modelo

Inicializamos el modelo de lenguaje que usará el agente para razonar y generar respuestas. Usamos `Gemini 2.5 Flash` de Google, un modelo rápido y eficiente. La temperatura a 0 garantiza respuestas deterministas, es decir, siempre consistentes ante la misma pregunta.

In [ ]:
# Inicializamos el modelo Gemini 2.5 Flash con temperatura 0
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

## 2. Memoria

Inicializamos el sistema de memoria del agente usando `MemorySaver`, que guarda el historial de la conversación en RAM. Esto permite que el agente recuerde el contexto entre turnos, sabiendo lo que se ha dicho anteriormente. La memoria se pierde al reiniciar el kernel, ya que vive únicamente en la memoria del proceso.

In [ ]:
# Memoria
memory = MemorySaver() # Guarda el historial de la conversación en la RAM

# Si quisieras guardar las conversaciones en una base de datos harías:
# from langgraph.checkpoint.sqlite import SqliteSaver
# memory = SqliteSaver.from_conn_string("conversaciones.db")

## 3. Tools

Las tools son las capacidades que le damos al agente para interactuar con el mundo exterior. En este caso definimos dos: una tool personalizada en Python para hacer cálculos matemáticos, y una tool externa que conecta con la API de Tavily para buscar información en internet en tiempo real. El agente decide automáticamente cuál usar en función de la pregunta del usuario.

### Tool personalizada: Calculadora

Creamos nuestra propia tool en Python. Es una función normal que el agente puede llamar cuando necesita hacer cálculos. Usamos `eval()` para ejecutar la expresión matemática, limitando las funciones disponibles por seguridad para evitar que se ejecute código malicioso.

In [ ]:
def calculadora(expresion: str) -> str: # Recibe una expresión matemática como string
    try:
        # eval() ejecuta la expresión matemática
        # __builtins__: {} bloquea funciones peligrosas del sistema como open(), import(), exec()
        resultado = eval(expresion, {"__builtins__": {}}, {}) # Para producción usar mejor numexpr o ast.literal_eval
        return f"El resultado es: {resultado}" # Devuelve el resultado como String
    except Exception as e:
        return f"Error en el cálculo: {str(e)}" # Si falla devuelve el error

### Tool externa: Búsqueda web con Tavily

Conectamos el agente con la API de Tavily para buscar información en internet en tiempo real. A diferencia de la calculadora, esta tool no la programamos nosotros, sino que la importamos directamente de la librería `langchain-tavily`. `max_results=3` limita el número de resultados que devuelve la búsqueda.

In [ ]:
# Tool externa para búsqueda web en tiempo real usando la API de Tavily (NO HACE FALTA EJECUTAR CON MCP)
tool_buscar = TavilySearch (max_results=3)

### Lista de tools
Combinamos las dos tools en una lista que le pasaremos al agente. El wrapper `Tool` permite darle nombre y descripción a la calculadora para que el agente sepa cuándo usarla.

### Opción A (SIN MCP)

Si usamos MCP, no ejecutar esta parte. Las tools se definen directamente en el código. Cada vez que quieras añadir una nueva tool tienes que programarla o importarla manualmente y añadirla a la lista. Es el enfoque clásico.

In [ ]:
# Definimos las tools que usará el modelo (NO HACE FALTA EJECUTAR CON MCP)
tools = [tool_buscar, Tool(name="calculadora", func=calculadora,
         description="Para cálculos matemáticos. Input: expresión como 15*23")]



### Opción B (CON MCP)

MCP (Model Context Protocol) es un protocolo estándar que permite a un agente conectarse a servidores externos y descubrir sus herramientas de forma dinámica, sin programar cada integración manualmente. Es como un USB-C para tools: conectas el agente al servidor y automáticamente sabe qué puede usar.

In [ ]:
!pip install -q langchain-mcp-adapters mcp

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

# Configuramos el servidor MCP de Tavily, Github y filesystem (como stdio, pero hay http también)
# Algunos clientes MCP no admiten servidores MCP basados ​​en SSE, hay que usar mcp-remote
# Mismo resultado que la semana pasada pero ahora las tools
# las descubre el agente automaticamente desde el servidor MCP.
# Necesitas Node.js instalado, compruebalo con node --version
client = MultiServerMCPClient(
    {
        "filesystem": {
            "transport": "stdio",
            "command": "npx",
            "args": ["-y", "@modelcontextprotocol/server-filesystem", "ruta_que_quieras_en_tu_pc"]
        },
        "tavily": {
            "transport": "stdio",
            "command": "npx",
            "args": ["-y", "tavily-mcp"],
            "env": {"TAVILY_API_KEY": os.environ["TAVILY_API_KEY"]}
        }
    }
)

# El agente pregunta al servidor ¿que tools tienes disponibles?
# Las descubre automaticamente, no las definimos nosotros
tools_mcp = await client.get_tools()
# Añadimos nuestra tool propia a las que vienen del servidor MCP
tools_mcp.append(Tool(name="calculadora", func=calculadora, description="Para cálculos matemáticos. Input: expresión como 15*23"))
# Listamos las tools
print(f"Tools cargadas desde MCP: {[t.name for t in tools_mcp]}")


## 4. Agente

Creamos el agente pasándole el modelo, las tools y la memoria. `create_react_agent` monta internamente el grafo ReAct que gestiona el ciclo de razonamiento y ejecución de tools.

In [ ]:
# Agente con memoria integrada
# agente = create_react_agent(llm, tools, checkpointer=memory)

agente_mcp = create_react_agent(llm, tools_mcp, checkpointer=memory)

## 5. Conversación

Iniciamos el chat en un bucle continuo donde el usuario puede escribir mensajes y el agente responde usando su memoria y tools. Cada mensaje se envía con el `thread_id` para que el agente mantenga el contexto de la conversación. Escribe "salir" para terminar.

In [ ]:
# Identificador del hilo de conversación, cada thread_id mantiene su propia memoria independiente

config = {"configurable": {"thread_id": "sesion_1"}}


# Bucle de la conversación, se ejecuta hasta que escribo salir
while True:
    mensaje = input("Tu: ")
    if mensaje.lower() == "salir":
        break

    #response = agente.invoke({"messages": [HumanMessage(mensaje)]}, config)

    #OPCIÓN MCP (descomentar y comentar arriba)
    response = await agente_mcp.ainvoke({"messages": [HumanMessage(mensaje)]},config)

    # Gemini a veces devuelve el contenido como lista y otras como string
    content = response["messages"][-1].content

    if isinstance(content, list):
      print(f"Agente: {content[0]['text']}")
    else:
      print(f"Agente: {content}")

    # print(f"Agente: {response['messages'][-1].content}")



Tu: Hola me llamo Hugo
Agente: Hola Hugo, ¿en qué puedo ayudarte?
Tu: Busca las 10 noticias mas importantes de IA
Agente: Aquí tienes 10 de las noticias más importantes sobre IA, basándome en la información más reciente disponible:

1.  **Inversión masiva en infraestructura de IA:** xAI construirá un centro de datos de $20 mil millones en Mississippi.
2.  **Preocupaciones éticas y de seguridad con la IA:** Grok AI está bajo escrutinio por generar imágenes explícitas, y un vehículo Waymo sin conductor atropelló a un niño en California.
3.  **Impacto de la IA en el empleo:** El auge de los centros de datos de IA está creando más oportunidades de trabajo para electricistas.
4.  **Desafíos sociales y económicos de la IA:** Agricultores en Maryland están luchando contra las compañías eléctricas debido al auge de la IA, y la creciente demanda de IA está causando escasez de RAM.
5.  **IA en el entretenimiento y la propiedad intelectual:** Disney ha llegado a un acuerdo con OpenAI para acceder